# Board Game Insights: Data Exploration & Feature Engineering
Welcome! This notebook explores a dataset from BoardGameGeek (BGG), featuring all ranked games in their database as of February 2021.

My goal is to turn messy, real-world data into actionable insights. It sounds just like what I do at work, but I digress! For some reason, using board game data is more fun. This exploratory analysis will lay the groundwork for visualizations and future machine learning models.<br>

---

**Project Roadmap:**

* **Chapter 1. Data Ingestion:** Explore the data in its raw form, establish parsing rules, and define schemas
* **Chapter 2. Data Cleaning:** Drop duplicates, handle nulls, and standardize text features
* **Chapter 3. Exploratory Data Analysis (EDA):** Uncover insights and have fun with visualization libraries!
* **Chapter 4. Feature Engineering:** Prepare the dataset for BI tools and/or predictive modeling

---

Thanks for joining me in this valiant effort. Let's see what we can cook up!


# Chapter 1: Data Ingestion
## Setup and Initialization

In [1]:
# import packages that give me workable data
import pandas as pd
import numpy as np

# import packages that give me pretty data
pd.set_option('display.max_columns', None)     # Remove column limit in displays
pd.set_option('display.max_colwidth', None)    # Remove max column width (default 50 characters max)
from IPython.display import display, Markdown  # Allows me to use Markdown logic as code in my output

# Setup Constants
FILE_PATH = "bgg_dataset.csv" 

# Establish path
import pathlib
PATH = pathlib.Path(r"C:\Users\kyrst\OneDrive\Documents\DataAnalyticsPortfolio\bgg")

# Import CSV
df = pd.read_csv(PATH / FILE_PATH, sep=";") # sep is needed since this dataset separates columns with ;

# view sample
print(f"# Columns: {len(df.columns):,}")
print(f"# Rows: {len(df):,}")
print(f"# Unique Board Games: {df['ID'].nunique():,}")
df.head()

# Columns: 14
# Rows: 20,343
# Unique Board Games: 20,327


,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,Mechanics,Domains
0,174430.0,Gloomhaven,2017.0,1,4,120,14,42055,"8,79",1,"3,86",68323.0,"Action Queue, Action Retrieval, Campaign / Battle Card Driven, Card Play Conflict Resolution, Communication Limits, Cooperative Game, Deck Construction, Deck Bag and Pool Building, Grid Movement, Hand Management, Hexagon Grid, Legacy Game, Modular Board, Once-Per-Game Abilities, Scenario / Mission / Campaign Game, Simultaneous Action Selection, Solo / Solitaire Game, Storytelling, Variable Player Powers","Strategy Games, Thematic Games"
1,161936.0,Pandemic Legacy: Season 1,2015.0,2,4,60,13,41643,"8,61",2,"2,84",65294.0,"Action Points, Cooperative Game, Hand Management, Legacy Game, Point to Point Movement, Set Collection, Trading, Variable Player Powers","Strategy Games, Thematic Games"
2,224517.0,Brass: Birmingham,2018.0,2,4,120,14,19217,"8,66",3,"3,91",28785.0,"Hand Management, Income, Loans, Market, Network and Route Building, Score-and-Reset Game, Tech Trees / Tech Tracks, Turn Order: Stat-Based, Variable Set-up",Strategy Games
3,167791.0,Terraforming Mars,2016.0,1,5,120,12,64864,"8,43",4,"3,24",87099.0,"Card Drafting, Drafting, End Game Bonuses, Hand Management, Hexagon Grid, Income, Set Collection, Solo / Solitaire Game, Take That, Tile Placement, Turn Order: Progressive, Variable Player Powers",Strategy Games
4,233078.0,Twilight Imperium: Fourth Edition,2017.0,3,6,480,14,13468,"8,70",5,"4,22",16831.0,"Action Drafting, Area Majority / Influence, Area-Impulse, Dice Rolling, Follow, Grid Movement, Hexagon Grid, Modular Board, Trading, Variable Phase Order, Variable Player Powers, Voting","Strategy Games, Thematic Games"


**For fun, let's view another sample. This time, make it random.**
- df.head() shows the first 5 rows by default, which are usually cleaner due to getting more attention
- Note that this random sample has NaN values

In [2]:
display(df.sample(5))

,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,Mechanics,Domains
4901,3412.0,Brandywine,2000.0,2,2,240,12,193,"7,27",4903,"2,67",665.0,Hexagon Grid,Wargames
3721,230089.0,Okanagan: Valley of the Lakes,2017.0,2,4,45,10,501,"6,76",3723,"2,17",864.0,"Area Majority / Influence, Enclosure, Set Collection, Tile Placement",NaN
6153,161782.0,Kremlin (Third Edition),2014.0,2,6,90,12,153,"6,95",6155,"2,75",526.0,"Hand Management, Secret Unit Deployment, Voting",NaN
944,283863.0,The Magnificent,2019.0,1,4,90,14,1575,"7,62",945,"3,14",2502.0,"Card Drafting, Contracts, Dice Rolling, Drafting, End Game Bonuses, Grid Coverage, Movement Points, Set Collection, Solo / Solitaire Game, Square Grid, Tile Placement, Track Movement, Turn Order: Stat-Based, Variable Player Powers",Strategy Games
9739,105864.0,On the Cards,2011.0,2,6,30,10,82,"6,55",9741,"1,56",162.0,Trick-taking,NaN


## Initial Inspection
- First, let's get a column summary using using df.info()
- Next, we'll see what quick fixes are possible
- Finally, the super cool exciting stuff: null and duplicate handling!

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20343 entries, 0 to 20342
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID                  20327 non-null  float64
 1   Name                20343 non-null  object 
 2   Year Published      20342 non-null  float64
 3   Min Players         20343 non-null  int64  
 4   Max Players         20343 non-null  int64  
 5   Play Time           20343 non-null  int64  
 6   Min Age             20343 non-null  int64  
 7   Users Rated         20343 non-null  int64  
 8   Rating Average      20343 non-null  object 
 9   BGG Rank            20343 non-null  int64  
 10  Complexity Average  20343 non-null  object 
 11  Owned Users         20320 non-null  float64
 12  Mechanics           18745 non-null  object 
 13  Domains             10184 non-null  object 
dtypes: float64(3), int64(6), object(5)
memory usage: 2.2+ MB


**Conclusions**
- `ID`, `Year Published` and `Owned Users` don't need to be floats. Later, we'll turn ID into a string since it's not meant to be used for math and is simply a randomly generated number. The other two fields should be integers
- `Rating Average` and `Complexity Average` use commas instead of decimal points, which makes pandas interpret them as objects
- `Name`, `Mechanics`, and `Domains` are objects that should be strings (due to having nulls)
- The combination of commas replacing decimal points and a semicolon column separator tells us that this is likely a European dataset. That's useful for data cleaning and provides supporting evidence for who to pressure to play board games with me in my personal life.

We can fix some of this by importing the dataset again and establishing some rules up front!

## The "Smart" Load
Let's re-read the dataset and establish more rules up front

In [4]:
# Establish custom data types for the problem children using a dictionary:
# Note: Using capital letters (ex: "Int64" vs. "int64") uses the nullable version
# This is useful because standard integers fail if they encounter a null value (nulls prefer floats)
custom_dtypes = {
    'ID': 'string',
    'Year Published': 'Int64',
    'Owned Users': 'Int64'
}

# Ingest data
df = pd.read_csv(
    PATH / FILE_PATH,     # Remember when we established these constants up front?
    sep=";",              # Tells pandas to look for semicolons as the dataset separator
    decimal=",",          # Fixes the '8,5' vs '8.5' issue automatically
    engine="pyarrow",     # Uses the fast arrow engine for reading
    dtype=custom_dtypes,  # Uses the custom dictionary above 
    
    # Let numpy determine the column type for remaining columns.
    # "nullable" tells numpy to keep a column's native type.
    # Otherwise, a single pd.NA would lead the whole column to become a float.
    dtype_backend="numpy_nullable"                                                                          
)

df.head()

,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,Mechanics,Domains
0,174430,Gloomhaven,2017,1,4,120,14,42055,8.79,1,3.86,68323,"Action Queue, Action Retrieval, Campaign / Battle Card Driven, Card Play Conflict Resolution, Communication Limits, Cooperative Game, Deck Construction, Deck Bag and Pool Building, Grid Movement, Hand Management, Hexagon Grid, Legacy Game, Modular Board, Once-Per-Game Abilities, Scenario / Mission / Campaign Game, Simultaneous Action Selection, Solo / Solitaire Game, Storytelling, Variable Player Powers","Strategy Games, Thematic Games"
1,161936,Pandemic Legacy: Season 1,2015,2,4,60,13,41643,8.61,2,2.84,65294,"Action Points, Cooperative Game, Hand Management, Legacy Game, Point to Point Movement, Set Collection, Trading, Variable Player Powers","Strategy Games, Thematic Games"
2,224517,Brass: Birmingham,2018,2,4,120,14,19217,8.66,3,3.91,28785,"Hand Management, Income, Loans, Market, Network and Route Building, Score-and-Reset Game, Tech Trees / Tech Tracks, Turn Order: Stat-Based, Variable Set-up",Strategy Games
3,167791,Terraforming Mars,2016,1,5,120,12,64864,8.43,4,3.24,87099,"Card Drafting, Drafting, End Game Bonuses, Hand Management, Hexagon Grid, Income, Set Collection, Solo / Solitaire Game, Take That, Tile Placement, Turn Order: Progressive, Variable Player Powers",Strategy Games
4,233078,Twilight Imperium: Fourth Edition,2017,3,6,480,14,13468,8.7,5,4.22,16831,"Action Drafting, Area Majority / Influence, Area-Impulse, Dice Rolling, Follow, Grid Movement, Hexagon Grid, Modular Board, Trading, Variable Phase Order, Variable Player Powers, Voting","Strategy Games, Thematic Games"


**Confirm it worked:**

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20343 entries, 0 to 20342
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID                  20327 non-null  string 
 1   Name                20343 non-null  string 
 2   Year Published      20342 non-null  Int64  
 3   Min Players         20343 non-null  Int64  
 4   Max Players         20343 non-null  Int64  
 5   Play Time           20343 non-null  Int64  
 6   Min Age             20343 non-null  Int64  
 7   Users Rated         20343 non-null  Int64  
 8   Rating Average      20343 non-null  Float64
 9   BGG Rank            20343 non-null  Int64  
 10  Complexity Average  20343 non-null  Float64
 11  Owned Users         20320 non-null  Int64  
 12  Mechanics           18745 non-null  string 
 13  Domains             10184 non-null  string 
dtypes: Float64(2), Int64(8), string(4)
memory usage: 2.4 MB


# Chapter 2: Data Cleaning
## Handling Duplicates
- We'll use .duplicated() to check for `ID` duplicates (since it's the primary key) and row duplicates
- **Conclusion:** No dups! However, there are rows with the same `Name` value (more on that below!)

In [6]:
# Check for entire rows
df[df.duplicated()] # No rows are entirely the same

,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,Mechanics,Domains


In [7]:
# Check for the primary key specifically
df[df.duplicated(subset=['ID'])] # All are NULL! We will likely remove these next.

,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,Mechanics,Domains
10835,<NA>,Die Erben von Hoax,1999,3,8,45,12,137,6.05,10837,2.0,<NA>,<NA>,<NA>
11152,<NA>,Rommel in North Africa: The War in the Desert 1941-42,1986,2,2,0,12,53,6.76,11154,4.0,<NA>,<NA>,<NA>
11669,<NA>,Migration: A Story of Generations,2012,2,4,30,12,49,7.2,11671,2.0,<NA>,<NA>,<NA>
12649,<NA>,Die Insel der steinernen Wachter,2009,2,4,120,12,49,6.73,12651,3.0,<NA>,<NA>,<NA>
12764,<NA>,Dragon Ball Z TCG (2014 edition),2014,2,2,20,8,33,7.03,12766,2.5,<NA>,<NA>,<NA>
13282,<NA>,Dwarfest,2014,2,6,45,12,82,6.13,13284,1.75,<NA>,<NA>,<NA>
13984,<NA>,Hus,<NA>,2,2,40,0,38,6.28,13986,2.0,<NA>,<NA>,<NA>
14053,<NA>,Contrario 2,2006,2,12,0,14,37,6.3,14055,1.0,<NA>,<NA>,<NA>
14663,<NA>,Warage: Extended Edition,2017,2,6,90,10,49,7.64,14665,3.0,<NA>,<NA>,<NA>
16292,<NA>,Rainbow,2010,2,6,20,8,35,5.65,16294,2.0,<NA>,<NA>,<NA>


In [8]:
# Check for game names. I wonder if some names may be shared across different versions.
df_shared_names = df[df.duplicated(subset=['Name'], keep=False)].sort_values(by='Name')

# Let's see some summary stats
print(f"Total # rows with the same name: {len(df_shared_names)}")
print(f"Total # unique names in subset: {df_shared_names['Name'].nunique()}")
print(f"Average # editions per unique name in subset: {len(df_shared_names) / df_shared_names['Name'].nunique():.2f}")
print(f"Total % of dataset rows from subset: {len(df_shared_names)/len(df):.1%}")
print(f"Total % of unique names from subset: {df_shared_names['Name'].nunique()/df['Name'].nunique():.1%}")

df_shared_names.head(6)

Total # rows with the same name: 690
Total # unique names in subset: 323
Average # editions per unique name in subset: 2.14
Total % of dataset rows from subset: 3.4%
Total % of unique names from subset: 1.6%


,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,Mechanics,Domains
14684,205495,1001,2016,2,8,45,8,55,6.11,14686,2.0,110,"Action Queue, Grid Movement, Simultaneous Action Selection, Voting",<NA>
13930,17183,1001,1900,2,4,60,12,31,6.43,13932,2.0,15,"Auction/Bidding, Trick-taking",<NA>
9693,11284,1862,2000,4,7,360,12,46,7.35,9695,3.71,72,"Network and Route Building, Stock Holding, Tile Placement, Trading",<NA>
19215,4456,1862,1990,1,2,180,12,51,4.68,19217,2.38,170,"Dice Rolling, Events, Hexagon Grid, Simulation",Wargames
19338,8308,3D Labyrinth,2002,2,4,30,5,138,5.13,19340,1.06,359,Modular Board,Children's Games
14751,274205,3D Labyrinth,2019,2,4,30,7,37,6.1,14753,1.5,82,"Modular Board, Network and Route Building, Point to Point Movement",<NA>


**Conclusions**

- There ARE in fact duplicates for `Name`, BUT other column values can differ. Since I see different `Year Published` values for each duplicate in this sample, I assume that duplicates exist for different editions of the game. However, we'll be sure to confirm that!
- On average, there are slightly over 2 editions per game in this subset. I wonder which games have the most editions?
- From the examples, multiple factors could change across editions:
    - `Min/Max Players` and `Min Age` differences indicate real game changes across editions
    - `Domains` changes could simply come from varying data quality across editions (ex: NULLs in one version but not the other)
- Overall, games with multiple editions account for 3.4% of the entire dataset's rows and 1.6% of its unique names

**Considerations**

Next, I'll answer as many of the above questions as I can. These are the next steps I'm considering:
1. Keep all duplicate `Name` rows, but add a new column for `Full Title` that concatenates `Name` with `Year Published`
2. Remove all but one duplicate `Name` and determine the best method for selection, likely based on max `Year Published` or `Owned Users` (or removing versions with NULLs)

Option 1 works better if we'd like to consider each edition as its own game. Option 2 works better if we're concerned about overweighting any features of duplicates. BUT since I personally know that different editions can vary wildly AND these rows only constitute 3.4% of the entire dataset, I'm leaning towards option 1.

In [9]:
# Create a dedicated view for shared titles
shared_titles = df[df.duplicated(subset=['Name'], keep=False)].sort_values("Name")

# Group by name to show a summary of the 'Collision' stats
duplicate_summary = shared_titles.groupby('Name').agg({
    'Year Published': ['min', 'max'],
    'ID': 'count',
    'Owned Users': 'sum'
}).rename(columns={'count': 'Version Count', 'sum': 'Total Owners'})

# Top 10 most 'duplicated' names
duplicate_summary.sort_values(('ID', 'Version Count'), ascending=False).head(10)

Year Published                  ID  Owned Users
                               min   max Version Count Total Owners
Name                                                               
Robin Hood                    1990  2019             6         1124
Gangster                      1985  2007             4         1220
Gettysburg                    1958  2018             4         3471
Chaos                         1970  2010             4          934
Cosmic Encounter              1977  2008             4        42018
Saga                          1980  2011             4         3445
Druids                        2004  2017             3          871
Clue                          1949  2013             3        35275
Battle of the Sexes           1986  2012             3         2188
Tortuga                       2003  2014             3         1461

## Missing Values (NULLs)
- First, we'll count nulls per column
- Next, we'll inspect samples of the dataset where each column has nulls

In [10]:
# count nulls per column
null_counts = df.isnull().sum().sort_values(ascending=False)

# Filter on columns with nulls
null_counts = null_counts[null_counts > 0]

# Let's turn this series into a dataframe! Why, you ask? She deserves to feel pretty.
display_nulls = (null_counts
    .reset_index(name='Null Count')
    .rename(columns={'index':'Column Name'})
                )

display(Markdown("#### Null count per column with NULLs"))
display(display_nulls)

#### Null count per column with NULLs

,Column Name,Null Count
0,Domains,10159
1,Mechanics,1598
2,Owned Users,23
3,ID,16
4,Year Published,1


> **Lesson learned from the code below:**
>
> 
> The code below worked in my local Jupyter Notebook, but the red highlights vanished on GitHub previews. This was how I learned that GitHub strips out custom CSS for security reasons.
>
> I've replaced the display_df code below with a version that allows me to see red cells while developing while also forcing GitHub to explicitly write `NaN`. 
>
> For future reference, here is the original code that only works locally:

```python
# This renders red nulls locally, but completely blank cells on GitHub
display_df = df[df[col].isnull()].head().style.highlight_null(color='red')
display(display_df)

In [11]:
# Inspect samples of the dataset where each column has nulls

def inspect_nulls(df):
    # First, list the columns with nulls
    null_counts = df.isnull().sum()
    cols_with_nulls = null_counts[null_counts > 0].index

    # Add condition for no nulls
    if len(cols_with_nulls) == 0:
        display(Markdown("#### Huzzah! No nulls found!"))
        return

    # Loop for null cols
    for col in cols_with_nulls:
        display(Markdown(f"---\n#### Top 5 rows where `{col}` is NULL"))
        display_df = (
            df[df[col].isnull()]
            .head()
            .style
            .format(na_rep="NaN") 
            .highlight_null(color='red')
        )
        display(display_df)

inspect_nulls(df)

---
#### Top 5 rows where `ID` is NULL

,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,Mechanics,Domains
10776,NaN,Ace of Aces: Jet Eagles,1990,2,2,20,10,110,6.260000,10778,2.000000,NaN,NaN,NaN
10835,NaN,Die Erben von Hoax,1999,3,8,45,12,137,6.050000,10837,2.000000,NaN,NaN,NaN
11152,NaN,Rommel in North Africa: The War in the Desert 1941-42,1986,2,2,0,12,53,6.760000,11154,4.000000,NaN,NaN,NaN
11669,NaN,Migration: A Story of Generations,2012,2,4,30,12,49,7.200000,11671,2.000000,NaN,NaN,NaN
12649,NaN,Die Insel der steinernen Wachter,2009,2,4,120,12,49,6.730000,12651,3.000000,NaN,NaN,NaN


---
#### Top 5 rows where `Year Published` is NULL

,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,Mechanics,Domains
13984,NaN,Hus,NaN,2,2,40,0,38,6.280000,13986,2.000000,NaN,NaN,NaN


---
#### Top 5 rows where `Owned Users` is NULL

,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,Mechanics,Domains
2828,202755,Guildhall Fantasy: Fellowship,2016,2,4,45,10,565,7.130000,2830,2.000000,NaN,"Hand Management, Take That, Set Collection",NaN
3590,196305,Guildhall Fantasy: Alliance,2016,2,4,45,10,360,7.200000,3592,2.140000,NaN,"Hand Management, Set Collection, Take That",NaN
3739,196306,Guildhall Fantasy: Coalition,2016,2,4,45,10,336,7.190000,3741,2.130000,NaN,"Hand Management, Set Collection, Take That",NaN
5807,289,Chariot Lords,1999,3,4,360,12,221,6.680000,5809,3.000000,NaN,"Area Movement, Variable Player Powers",NaN
9202,6813,Operation Market Garden: Descent into Hell,1985,2,2,120,12,94,6.720000,9204,3.000000,NaN,"Dice Rolling, Events, Grid Movement, Hexagon Grid, Hidden Movement, Movement Points, Ratio / Combat Results Table, Secret Unit Deployment, Simulation, Zone of Control",NaN


---
#### Top 5 rows where `Mechanics` is NULL

,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,Mechanics,Domains
1059,85256,Timeline: Inventions,2010,2,8,15,8,7257,6.710000,1060,1.110000,12448,NaN,Family Games
1150,113401,Timeline: Events,2011,2,8,15,8,4208,6.780000,1151,1.100000,7924,NaN,"Family Games, Party Games"
1216,131325,Timeline: Diversity,2012,2,8,15,8,3790,6.790000,1217,1.070000,7589,NaN,"Family Games, Party Games"
1343,99975,Timeline: Discoveries,2011,2,8,15,8,3506,6.730000,1344,1.180000,6855,NaN,Family Games
1397,145189,Timeline: Music & Cinema,2013,2,8,15,8,3090,6.750000,1398,1.100000,5986,NaN,"Family Games, Party Games"


---
#### Top 5 rows where `Domains` is NULL

,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,Mechanics,Domains
663,293141,King of Tokyo: Dark Edition,2020,2,6,30,8,1854,7.890000,664,1.630000,5157,"Card Drafting, Dice Rolling, King of the Hill, Player Elimination, Push Your Luck",NaN
1011,177802,Smash Up: It's Your Fault!,2016,2,2,60,14,1663,7.530000,1012,2.060000,6347,"Area Majority / Influence, Hand Management, Take That, Variable Phase Order",NaN
1047,226520,Exit: The Game - The Sinister Mansion,2018,1,4,90,12,1595,7.450000,1048,2.610000,3446,Cooperative Game,NaN
1073,274638,Unmatched: Robin Hood vs. Bigfoot,2019,2,2,20,9,924,8.240000,1074,1.930000,2665,"Action Points, Hand Management, Line of Sight, Point to Point Movement, Take That, Variable Player Powers",NaN
1092,215841,Exit: The Game - The Forgotten Island,2017,1,4,90,12,2736,7.020000,1093,2.700000,5438,Cooperative Game,NaN


**Conclusions**

Based on the samples above:
- nulls tend to come in groups for `ID`, `Year Published`, and `Owned Users`
- `Mechanics` and `Domains` are more often the only null value in a row

Based on the samples and counts, this is my conclusion for each column with missing values:
| Column Name | Null Count | Strategy |
| :--- | :--- | :--- |
| **Domains** | 10,159 | Fill with Null Indicator |
| **Mechanics** | 1,598 | Fill with Null Indicator |
| **Owned Users** | 23 | Drop Rows |
| **ID** | 16 | Drop Rows |
| **Year Published** | 1 | Drop Rows |

`Domains` and `Mechanics` have a lot of nulls. It could be random or the games with nulls could have more in common. Either way, let's track it by keeping these rows and imputing with something else.

Next, I'll explore `Domains` and `Mechanics` to understand them better before deciding on impute values and next steps

In [12]:
# Let's see how many unique values exist for Domains and Mechanis
check_unique = ['Domains', 'Mechanics']

for col in check_unique:
    print(f"{col}: {df[col].nunique()} unique values")


Domains: 39 unique values
Mechanics: 7381 unique values


Since `Domains` has a limited # of unique values, let's check them all out. 
- Conclusion: It's a mix of single value and comma delimited lists

In [13]:
display(Markdown("#### Value counts for `Domains`"))
display(df['Domains'].value_counts(dropna=False).reset_index())

#### Value counts for `Domains`

,Domains,count
0,<NA>,10159
1,Wargames,3029
2,Strategy Games,1455
3,Family Games,1340
4,Abstract Games,869
5,Children's Games,708
6,Thematic Games,647
7,Party Games,409
8,"Family Games, Strategy Games",354
9,Customizable Games,235


Since there are many unique values for `Mechanics`, let's look at the top and bottom values
- Conclusion: There are more lists, separated by a "," or a "/"

In [14]:
# The Top 10
display(Markdown("#### Top 10 most common values for `Mechanics`"))
display(df['Mechanics'].value_counts(dropna=False).head(10).reset_index())

# The Bottom 10
display(Markdown("#### Bottom 10 most common values for `Mechanics`"))
display(df['Mechanics'].value_counts(dropna=False).tail(10).reset_index())

#### Top 10 most common values for `Mechanics`

,Mechanics,count
0,<NA>,1598
1,Hand Management,432
2,Hexagon Grid,412
3,Dice Rolling,372
4,Roll / Spin and Move,369
5,Tile Placement,285
6,"Dice Rolling, Hexagon Grid, Simulation",260
7,"Dice Rolling, Hexagon Grid",256
8,Set Collection,231
9,"Hand Management, Set Collection",179


#### Bottom 10 most common values for `Mechanics`

,Mechanics,count
0,"Area Majority / Influence, Card Drafting, Modular Board",1
1,"Area Majority / Influence, Campaign / Battle Card Driven, Dice Rolling, Hand Management, Simulation, Tug of War",1
2,"Area Movement, Dice Rolling, Events, Race, Scenario / Mission / Campaign Game, Simultaneous Action Selection",1
3,"Area Majority / Influence, Dice Rolling, Push Your Luck, Take That",1
4,"Action Queue, Modular Board, Team-Based Game",1
5,"Dice Rolling, Measurement Movement, Pick-up and Deliver, Variable Player Powers, Variable Set-up",1
6,"Action Points, Dice Rolling, Grid Movement, Modular Board, Variable Phase Order, Variable Player Powers",1
7,"Area Movement, Hidden Movement, Secret Unit Deployment, Team-Based Game",1
8,"Auction/Bidding, Auction: Sealed Bid, Constrained Bidding, Pick-up and Deliver, Time Track, Worker Placement",1
9,"Dice Rolling, Grid Movement, Race, Roll / Spin and Move, Square Grid",1


**Conclusions**

Based on the investigation above, I'll simply fill nulls with 'Unknown' and then proceed with one-hot encoding later

In [15]:
# Drop rows based on nulls
drop_nulls = ['Owned Users', 'ID', 'Year Published']
df = df.dropna(subset = drop_nulls)

# Impute
df['Domains'] = df['Domains'].fillna("Unknown")
df['Mechanics'] = df['Mechanics'].fillna("Unknown")

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20320 entries, 0 to 20342
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID                  20320 non-null  string 
 1   Name                20320 non-null  string 
 2   Year Published      20320 non-null  Int64  
 3   Min Players         20320 non-null  Int64  
 4   Max Players         20320 non-null  Int64  
 5   Play Time           20320 non-null  Int64  
 6   Min Age             20320 non-null  Int64  
 7   Users Rated         20320 non-null  Int64  
 8   Rating Average      20320 non-null  Float64
 9   BGG Rank            20320 non-null  Int64  
 10  Complexity Average  20320 non-null  Float64
 11  Owned Users         20320 non-null  Int64  
 12  Mechanics           20320 non-null  string 
 13  Domains             20320 non-null  string 
dtypes: Float64(2), Int64(8), string(4)
memory usage: 2.5 MB


In [16]:
df.sample(5)

,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,Mechanics,Domains
8733,1404,Strand-Cup,2000,4,8,45,10,193,6.1,8735,1.2,337,Team-Based Game,Unknown
16703,77148,Globalization,2010,2,6,60,12,84,5.67,16705,2.57,206,"Auction/Bidding, Loans",Unknown
7368,223481,Take the Gold,2017,2,6,15,8,163,6.69,7370,1.0,407,"Memory, Take That",Unknown
6257,223777,Planet of the Apes,2017,1,4,90,14,151,7.1,6259,2.4,486,"Cooperative Game, Dice Rolling, Set Collection, Storytelling, Variable Player Powers",Unknown
13924,252530,JetLag,2018,3,8,25,12,42,6.22,13926,1.0,99,Memory,Unknown


## Text Standardization & Structural Fixes
Next, we'll investigate the `Mechanics` column deeper.
- Determine if "/" is meaningful or if it should be treated like a comma
- Check total # unique text values
- Bin/categorize
- Determine if low counts are noise and should be removed

# Chapter 3: Exploratory Data Analysis

<div id="chp4"></div>

# Chapter 4: Feature Engineering
## Binning

## Feature Creation

## One-Hot Encoding
Finally! I do this last because it makes the table wider, which adds more processing time and makes it more difficult for the human eye to read. For some reason it's just so satisying though.